## Identify Urban Heat Island (UHI) Risk with Raster Analysis

### Data overview
- 2024 Fractional Impervious Surface (CONUS): https://www.mrlc.gov/downloads/sciweb1/shared/mrlc/metadata/Annual_NLCD_FctImp_2024_CU_C1V1.xml
- Arizona state census block group: https://www.census.gov/cgi-bin/geo/shapefiles/index.php?year=2024&layergroup=Block+Groups
- Pheonix city boundary: https://maps.phoenix.gov/pub/rest/services/Public/CityBoundary/MapServer/0
- US cities: https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html

In [0]:
import geoanalytics

In [0]:
from geoanalytics.raster import functions as RT
from geoanalytics.sql import functions as ST
from pyspark.sql import functions as F

### 💡Idenity heat risk in City of Phoenix

In [0]:
# Load Phoenix census block groups as polygons
phoenix_block_groups = spark.read.format("shapefile") \
                            .load("abfss://{0}@{1}.dfs.core.windows.net/phoenix_block_group/".format(container,storage_account)) \
                            .withColumn("geometry", ST.transform("geometry", 5070))

In [0]:
# plot Phenoix census block group with basemap
phoenix_block_groups.st.plot(basemap="light")

#### Load Raster dataset

In [0]:
# get the extent of block groups in city of Phoenix
phoenix_extent = phoenix_block_groups.st.get_extent()

# load Fractional Impervious Surface raster
raster = spark.read.format("raster") \
            .load("abfss://{0}@{1}.dfs.core.windows.net/Fractional_Impervious_Surface/nlcd-fct-2024-sref.tif".format(container,storage_account)) \
            .withColumn("raster", RT.bbox_clip("raster", xmin=phoenix_extent.min_x, 
                                                         ymin=phoenix_extent.min_y, 
                                                         xmax=phoenix_extent.max_x, 
                                                         ymax=phoenix_extent.max_y)) \
            .where("raster is not NULL")

#### Inspect Raster properties

In [0]:
raster_object = raster.first().raster
print(f"Shape: {(raster_object.num_bands, raster_object.num_columns, raster_object.num_rows)}")
print(f"spatial_reference: {raster_object.spatial_reference}")
print(f"Pixel type: {raster_object.pixel_type}")

In [0]:
raster.agg(F.collect_list("raster").alias("tiles")) \
        .withColumn("agg_raster", RT.merge('tiles')) \
        .withColumn("extent", RT.info("agg_raster").extent) \
        .withColumn("stats", RT.band_statistics("agg_raster", 1)) \
        .selectExpr("extent", "stats.*").show(1, vertical=True, truncate=False)

#### Plot raster data with polygons

In [0]:
margin = 10000.0
plot_result = raster.rt.plot(cmap="YlOrRd", basemap="light", sr=5070,
                             extent=(phoenix_extent.min_x - margin, 
                                     phoenix_extent.min_y - margin, 
                                     phoenix_extent.max_x + margin, 
                                     phoenix_extent.max_y + margin), 
                             figsize=(7,7))
phoenix_block_groups.st.plot(ax=plot_result, basemap="light", facecolor="none", edgecolor="Blue", linewidth=0.5)

####Summarize FIS value into each block group

In [0]:
from geoanalytics.tools import ZonalStatistics

# Use Zonal Statistics to summarize Fractional Impervious Surface level into census block groups
result = ZonalStatistics() \
        .setZones(phoenix_block_groups) \
        .setZoneIdColumns("GEOID") \
        .setRasterColumn("raster") \
        .includeZoneGeometry(True)\
        .run(raster)

In [0]:
# Plot the census-block-level mean annual Fractional Impervious Surface level in city of Pheonix
result_plot = result.st.plot(cmap_values = "Mean",
                             legend=True,
                             legend_kwds={"orientation": "horizontal", "location": "bottom", "shrink": 0.7, "pad": 0.08},
                             figsize=(12,7),
                             basemap="light")

result_plot.set_title("Mean FIS of Pheonix census block groups")

### 💡Identify top 10 US cities with the highest UHI risk index

In [0]:
cities = spark.read.format("shapefile").load("abfss://{0}@{1}.dfs.core.windows.net/cities".format(container,storage_account)) \
              .groupBy("NAME")\
              .agg(ST.aggr_union("geometry").alias("city_geometry")) \
              .withColumnRenamed("NAME", "CITY_NAME")

In [0]:
us_census_block_group = spark.read.format("shapefile") \
                          .load("abfss://{0}@{1}.dfs.core.windows.net/us_census_block_group".format(container,storage_account)) \
                          .groupBy("ID", "NAME", "COUNTY", "STATE_NAME", "OBJECTID")\
                          .agg(ST.aggr_union("geometry").alias("bg_geometry"))

In [0]:
from geoanalytics.tools import Clip
from geoanalytics.tools import SpatiotemporalJoin

us_cities_block_groups = Clip().run(input_dataframe=us_census_block_group, clip_dataframe=cities) \
                      .drop("bg_geometry") \
                      .withColumn("bg_geometry", ST.transform("clip_geometry", 5070))

join_result = SpatiotemporalJoin() \
            .setLeftJoin(left_join=False) \
            .setJoinOneToMany()\
            .setSpatialRelationship(spatial_relationship="within") \
            .run(target_dataframe=us_cities_block_groups, join_dataframe=cities)

us_cities_block_groups_final = join_result.drop("city_geometry", "clip_geometry").persist()

#### Classify heat risk using RT_Apply

In [0]:
from pyspark.sql.functions import when

us_extent = us_cities_block_groups_final.st.get_extent()

raster = spark.read.format("raster") \
              .load("abfss://{0}@{1}.dfs.core.windows.net/Fractional_Impervious_Surface/nlcd-fct-2024-sref.tif" \
                    .format(container, storage_account))\
              .withColumn("raster", RT.bbox_clip("raster", xmin=us_extent.min_x, 
                                                           ymin=us_extent.min_y, 
                                                           xmax=us_extent.max_x, 
                                                           ymax=us_extent.max_y)) \
             .where("raster is NOT NULL") \
             .withColumn(
                "raster",
                RT.apply(
                    "raster",
                    1,
                    lambda px: when(px.value >= 70, 1).otherwise(0)
                )
             ).persist()

In [0]:
from geoanalytics.tools import ZonalStatistics

# Use Zonal Statistics to summarize Fractional Impervious Surface level into census block groups
result = ZonalStatistics() \
        .setZones(us_cities_block_groups_final) \
        .setZoneIdColumns("ID", "CITY_NAME") \
        .setRasterColumn("raster") \
        .includeZoneGeometry(False) \
        .includePercentiles(False) \
        .run(raster)

result.printSchema()

#### Aggregate results

In [0]:
result.groupBy("CITY_NAME").agg(F.sum("Sum").alias("area_of_high_UHI_risk")).sort("area_of_high_UHI_risk", ascending=False).show(10)